In [2]:

import sys
import os
import time
import numpy as np

import MDAnalysis as mda
# from MDAnalysis.analysis import align
# from MDAnalysis.analysis import distances

import westpa
from westpa.analysis import Run

import matplotlib.pyplot as plt

In [3]:
#this barely needs to be a method but having the .h5 stuff compartmentalized is nice
def load_h5_pcs(h5path, miniter, maxiter):
    
    run = Run.open(h5path)

    #set maximum iteration automatically
    if maxiter == -1:
        maxiter = run.num_iterations

    pcs = [iteration.pcoords for iteration in run if (iteration.number >= miniter and iteration.number < maxiter)]

    return pcs

In [5]:
#specify input file

cftr_west = "/home/jonathan/Documents/grabelab/cftr/chloe-data"
cftr_refpc = "/home/jonathan/Documents/grabelab/cftr/refeaturization"

h5paths_names = [[f"{cftr_west}/wstp_cftr_1_degrabo/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_1", "pyrazole-1", "blue"],
                  [f"{cftr_west}/wstp_cftr_2_wynton/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_2", "pyrazole-2", "cyan"],
                  [f"{cftr_west}/wstp_lip_glpg_1/west-040925.h5", f"{cftr_refpc}/lip_glpg_1", "undecanol-1", "red"],
                  [f"{cftr_west}/wstp_lip_glpg_2/west-040925.h5", f"{cftr_refpc}/lip_glpg_2", "undecanol-2", "orange"]]

#westpa rounds to load
minround = 0
maxround = -1

run_ind = 0

data_paths = ["/home/jonathan/Documents/grabelab/cftr/revisions/abbv-974-1"]
data_path = data_paths[run_ind]


In [6]:
pcs_all = load_h5_pcs(h5paths_names[run_ind][0], minround, maxround)

In [44]:
nbins = 51
binbounds = np.arange(0,nbins,1)
prot_wat_contacts_by_bin = [[] for a in range(nbins)]
prot_lig_contacts_by_bin = [[] for a in range(nbins)]
lig_prot_contacts_by_bin = [[] for a in range(nbins)]

#loop over WE rounds
for r in range(1,1000,10):
    
    #get progress coordinates of the walkers, accounting for the occasional corrupted file
    walkers = np.load(f"{data_path}/pc_data_round_{r}_obs_0_v1.npy")
    pcs_flat = pcs_all[r-1][:,-1].flatten()
    pcs = [pcs_flat[w] for w in walkers]

    bins = np.digitize(pcs, binbounds)

    #load water coordinates
    protein_water_contacts = np.load(f"{data_path}/pc_data_round_{r}_obs_2_v1.npy")
    protein_ligand_contacts = np.load(f"{data_path}/pc_data_round_{r}_obs_4_v1.npy")

    #get coordinates of waters in each bin
    for b, pwc, plc in zip(bins, protein_water_contacts, protein_ligand_contacts):
        prot_wat_contacts_by_bin[b].append(pwc)
        prot_lig_contacts_by_bin[b].append(np.where(np.sum(plc, axis = 1)>0, 1, 0))
        lig_prot_contacts_by_bin[b].append(np.where(np.sum(plc, axis = 0)>0, 1, 0))

for i, w in enumerate(prot_wat_contacts_by_bin):
    print(f"{i}: {len(w)}")

prot_wat_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in prot_wat_contacts_by_bin]
prot_lig_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in prot_lig_contacts_by_bin]
lig_prot_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in lig_prot_contacts_by_bin]



0: 0
1: 1377
2: 1807
3: 1436
4: 1069
5: 1128
6: 430
7: 241
8: 298
9: 277
10: 330
11: 249
12: 229
13: 141
14: 59
15: 29
16: 23
17: 10
18: 5
19: 2
20: 0
21: 0
22: 0
23: 0
24: 0
25: 0
26: 0
27: 0
28: 0
29: 0
30: 0
31: 0
32: 0
33: 0
34: 0
35: 0
36: 0
37: 0
38: 0
39: 0
40: 0
41: 0
42: 0
43: 0
44: 0
45: 0
46: 0
47: 0
48: 0
49: 0
50: 0


In [45]:
print(len(prot_lig_contacts_by_bin[1]))
print(len(lig_prot_contacts_by_bin[1]))


3000
24


In [46]:
def tmd_query():
    segment_resis = [[77, 149], [192, 245], [298, 362], [988, 1034], [857, 889], [900, 942], [1094, 1154]]
    #print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

In [47]:
#Chatgpt prompt:
"""write a python function that takes as its arguments a path to a pdb file, an mdanalysis atomgroup, and a list of numbers equal in length to the atom group, 
and saves a copy of the pdb file with the b factors of the atoms in the atom group set equal to the numbers in the list. 
Match atoms using residue names and numbers since the indices do not match between the atomgroup and pdb file"""


def write_bfactors_by_residue_match(pdb_path, atomgroup, values, output_pdb=None):
    """
    Write a copy of a PDB file with B-factors set for atoms matching an AtomGroup.
    Matching is done using (resname, resid, atom name), NOT indices.

    Parameters
    ----------
    pdb_path : str
        Path to input PDB file.
    atomgroup : MDAnalysis AtomGroup
        AtomGroup derived from (possibly different) structure.
    values : array-like
        Numbers equal in length to atomgroup.
    output_pdb : str
        Path to output PDB file.
    """

    if output_pdb is None:
        base = os.path.splitext(pdb_path)[0]
        output_pdb = base + "_bfactored.pdb"

    values = np.asarray(values)

    if len(atomgroup) != len(values):
        raise ValueError("Length of values must equal AtomGroup length.")

    # Load fresh universe from PDB to preserve original ordering
    u = mda.Universe(pdb_path)

    # Ensure tempfactors exist
    if not hasattr(u.atoms, "tempfactors"):
        u.add_TopologyAttr("tempfactors")

    # Start from existing B-factors (or zeros)
    try:
        b = u.atoms.tempfactors.copy()
    except AttributeError:
        b = np.zeros(len(u.atoms))

    # Build lookup dictionary from PDB universe
    pdb_lookup = {}
    for atom in u.atoms:
        key = (atom.resname, atom.resid, atom.name)
        pdb_lookup[key] = atom.index

    # Assign B-factors by matching keys
    for atom, value in zip(atomgroup, values):
        key = (atom.resname, atom.resid, atom.name)

        if key not in pdb_lookup:
            raise ValueError(
                f"Atom not found in PDB: resname={atom.resname}, "
                f"resid={atom.resid}, name={atom.name}"
            )

        pdb_index = pdb_lookup[key]
        b[pdb_index] = value

    # Reassign full array (required by MDAnalysis)
    u.atoms.tempfactors = b

    # Write output
    with mda.Writer(output_pdb, multiframe=False) as W:
        W.write(u.atoms)


In [48]:
def ref_prot_seles():
    ref_frame_path = '/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro'
    u = mda.Universe(ref_frame_path)
    prot_sel = u.select_atoms(f"{tmd_query()} and not name H*")
    lig_sel = u.select_atoms("resname LJP and not name H*")
    return u, prot_sel, lig_sel

u, prot_sel, lig_sel = ref_prot_seles()

In [50]:
bin = 10
pwc = prot_wat_contacts_by_bin[bin]
plc = prot_lig_contacts_by_bin[bin]
lpc = lig_prot_contacts_by_bin[bin]

write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, pwc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_pwc.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", lig_sel, lpc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_lpc.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, plc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_plc.pdb")

3

In [ ]:

#                                                     END 

In [84]:
import numpy as np
import plotly.graph_objects as go

# Example large dataset
# N = 200000
# coords = np.random.randn(N,3)
# x = coords[:,0]
# y = coords[:,1]
# z = coords[:,2]

mean_prot = np.mean(prot_pos, axis = 0)

bin = 5
interval=1

#print(waters_by_bin[bin])

x = waters_by_bin[bin][::interval,0]-mean_prot[0]
y = waters_by_bin[bin][::interval,1]-mean_prot[1]
z = waters_by_bin[bin][::interval,2]-mean_prot[2]

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        hoverinfo="skip",
        marker=dict(
            size=3,
            opacity=0.7,
            color="red"#z,          # GPU color mapping
            #colorscale="Viridis"
        )
    )
)


if True:
    xp = prot_pos[:,0]-mean_prot[0]
    yp = prot_pos[:,1]-mean_prot[1]
    zp = prot_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xp,
            y=yp,
            z=zp,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="black"
            )
        )
    )

    xl = lig_pos[:,0]-mean_prot[0]
    yl = lig_pos[:,1]-mean_prot[1]
    zl = lig_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xl,
            y=yl,
            z=zl,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="blue"
            )
        )
    )

fig.update_layout(
    title="Large Interactive 3D Scatter",
    scene=dict(
        xaxis = dict(nticks=4, range=[-30,30],),
        yaxis = dict(nticks=4, range=[-30,30],),
        zaxis = dict(nticks=4, range=[-30,30],),
        aspectratio=dict(x=1, y=1, z=1),
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=1000,
    height=1000,
    margin=dict(l=0,r=0,b=0,t=40),
    scene_camera=dict(
            eye=dict(x=3, y=3, z=3)
        )

)

fig.show(config={"scrollZoom": True})